# Territorial.io — Neural Bot TRAINING (GPU)One session: **GPU trains**, **CPU workers feed sim data**. Stages: collect (sim) -> vision -> clone -> **real (fine-tune on REAL recorded matches from HF)** -> PPO (curriculum) -> eval -> export to HF.P100 handling: auto-installs torch 2.4.1+cu121 (sm_60 kernels); falls back to CPU (85k params). Internet ON. Loud failure: any stage rc!=0 stops the notebook.

In [ ]:
# 1. clone + deps + CUDA sanity!git clone -q https://github.com/amerameryou1-blip/bot.git /kaggle/working/bot || trueimport sys, os, subprocessos.chdir('/kaggle/working/bot')sys.path.insert(0, '/kaggle/working/bot/src')def cuda_ok():    code = "import torch; a=torch.randn(16,16,device='cuda'); (a@a).sum().item(); torch.cuda.synchronize(); print('CUDA_OK')"    r = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)    return 'CUDA_OK' in r.stdoutimport torchgpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'print('GPU:', gpu)if cuda_ok():    print('CUDA works with current torch')else:    print('CUDA broken (likely P100 sm_60) — installing torch 2.4.1+cu121...')    subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',                    'torch==2.4.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=False)    if cuda_ok():        print('CUDA OK with torch 2.4.1+cu121')    else:        print('torch 2.4.1 still not usable — FORCING CPU')        os.environ['FORCE_CPU'] = '1'subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow', 'numpy', 'pytesseract',                'huggingface_hub', 'safetensors'], check=False)print('device decision done; FORCE_CPU =', os.environ.get('FORCE_CPU', '0'))print('ready')

In [ ]:
# 2. config (env vars — train_nn.py reads them)import osos.environ.setdefault('COLLECT_SEEDS', '60')os.environ.setdefault('WORKERS', '4')os.environ.setdefault('PPO_ROUNDS', '100')print({k: os.environ[k] for k in ('COLLECT_SEEDS','WORKERS','PPO_ROUNDS','FORCE_CPU') if k in os.environ})

In [ ]:
# 3. COLLECT (CPU-parallel, teacher vs medium bots)import subprocess, sysdef run_stage(stage, extra=None):    args = [sys.executable, '-u', 'scripts/train_nn.py', stage]    if extra: args.append(str(extra))    r = subprocess.run(args, capture_output=True, text=True)    print(r.stdout[-3000:])    if r.returncode != 0:        print('STAGE FAILED:', r.stderr[-2000:])        raise RuntimeError('stage ' + stage + ' failed')    return rrun_stage('collect')

In [ ]:
# 3b. PULL + MERGE worker shards from HF (diverse maps)import os, subprocess, systok = os.environ.get('HF_TOKEN', '''')if tok:    r = subprocess.run([sys.executable,'-u','scripts/merge_worker_data.py'], capture_output=True, text=True, env=dict(os.environ, HF_TOKEN=tok))    print(r.stdout[-2000:]); print(r.stderr[-1000:] if r.returncode else '')else:    print('HF_TOKEN not set — training on local collect only')

In [ ]:
# 3c. PULL REAL MATCH RECORDINGS + LABEL THEM (real-data stage input)import os, subprocess, sys, glob, shutil, jsontok = os.environ.get('HF_TOKEN', '''')os.environ['HF_TOKEN'] = tokfrom huggingface_hub import snapshot_downloados.makedirs('recordings', exist_ok=True)try:    p = snapshot_download(repo_id='amer224/territorial-bot-data', repo_type='dataset',                          allow_patterns=['recordings/*'], token=tok)    print('recordings pulled to', p)    for m in glob.glob(os.path.join(p, 'recordings', '*', 'meta.json')):        sess = os.path.dirname(m)        dst = os.path.join('recordings', os.path.basename(sess))        if not os.path.exists(dst):            shutil.copytree(sess, dst)    sessions = sorted(glob.glob('recordings/*/meta.json'))    print('REAL SESSIONS:', len(sessions))    for s in sessions:        meta = json.load(open(s))        print('  ', os.path.basename(os.path.dirname(s)), 'frames=', meta.get('frames'),              'zoom=', meta.get('zoom_level'), 'cam_pass=', meta.get('camera_pass'))except Exception as e:    print('recordings pull failed (continuing on sim only):', e)os.makedirs('weights/nn', exist_ok=True)r = subprocess.run([sys.executable, '-u', 'scripts/label_real.py', '--recordings', 'recordings',                    '--out', 'weights/nn/real_vision.npz', '--save-anyway'], capture_output=True, text=True)print(r.stdout[-1500:])if r.returncode != 0:    print('label_real failed (continuing):', r.stderr[-1000:])else:    print('REAL LABELS READY')

In [ ]:
# 4. VISION + CLONE + REAL fine-tune (GPU)run_stage('vision')run_stage('clone')run_stage('real')

In [ ]:
# 5. PPO — learn to WIN (curriculum medium->hard)import osrun_stage('ppo', os.environ.get('PPO_ROUNDS', '100'))

In [ ]:
# 6. FINAL EVAL only (does NOT re-train)run_stage('eval')print('Model at /kaggle/working/bot/weights/nn/model.pt')

In [ ]:
# 7. EXPORT to Hugging Face (safetensors + config + model card)import os, subprocess, systok = os.environ.get('HF_TOKEN', '''')r = subprocess.run([sys.executable, '-u', 'scripts/export_hf.py'], capture_output=True, text=True,                   env=dict(os.environ, HF_TOKEN=tok))print(r.stdout[-2000:])if r.returncode != 0:    print('EXPORT FAILED:', r.stderr[-1000:])    raise RuntimeError('export failed')print('EXPORTED to amer224/territorial-bot-nn')